# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedosrf/flyrank-ml-internship-ahmedosrf/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook describes the Lane 2 slice used for content-refresh prioritization. I use a mid-panel month (`2026-03`) for development and keep the final-month sample sealed. All identifiers are pseudonymous and are used only for grouping and joins.

## 1. Unit of analysis + time window

**Contract answer.** One modeling row represents one pseudonymized **client–content item pair**. The raw warehouse grain is one report date × client × content item. For this first feature frame, features summarize the first 14 days of March 2026 (`2026-03-01` through `2026-03-14`), and the proxy outcome is measured in the later evaluation window (`2026-03-15` through `2026-03-31`). The three-day boundary is not used as a feature; it keeps the decision window separate from the outcome window.

**Lane decision.** The output is a score for which content items deserve a refresh review first. The proxy label is `is_declining_proxy`: later-window impressions are lower than the pre-decision-window impressions. This is directional decision support, not a claim that the page caused its own decline.

In [1]:
# Setup: use HF_TOKEN from Colab Secrets or the local environment. Never hard-code it.
import os
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError('Add a Hugging Face Read token as HF_TOKEN in Colab Secrets or the environment.')

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=2")
con.execute("SET preserve_insertion_order=false")
con.execute('INSTALL httpfs')
con.execute('LOAD httpfs')
_safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{_safe_token}')")
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
print('Connected to the March 2026 warehouse partition; token value is not displayed.')

Connected to the March 2026 warehouse partition; token value is not displayed.


## 2. Fields: feature / label / context / excluded

**Features (five, and only five in the first frame).** `gsc_impressions_pre` is knowable before the decision because it is aggregated from March 1–14; `gsc_clicks_pre` is available in the same window; `gsc_avg_position_pre` summarizes observed Search Console position in the same window; `ga4_sessions_pre` is available before the decision when Analytics data is available; and `ga4_engaged_sessions_pre` is the corresponding engaged-session count. The GA4 fields remain missing when `ga4_data_available` is not true; I do not silently turn missing Analytics coverage into observed zero activity.

**Label / proxy.** `is_declining_proxy` is computed only from the later March 15–31 evaluation window compared with the pre-decision window. It is an outcome proxy, never a feature.

**Context.** `client_hash_id`, `content_hash_id`, `pre_ga4_available`, and the window dates are used for grouping, auditing, and later grouped splitting only. Identifiers are not model inputs.

**Excluded.** `trend_direction`, `trend_pct`, or any future-period metric is excluded because it is either label-derived or unavailable at the decision moment. `month` is also excluded because it is a partition/audit field, not page behavior. Raw query text is not present in this table and no private client identity is inferred.

In [2]:
# The five feature names are explicit so the model frame cannot silently grow.
FEATURES = [
    'gsc_impressions_pre',
    'gsc_clicks_pre',
    'gsc_avg_position_pre',
    'ga4_sessions_pre',
    'ga4_engaged_sessions_pre',
]
print('Five allowed feature columns:', FEATURES)

Five allowed feature columns: ['gsc_impressions_pre', 'gsc_clicks_pre', 'gsc_avg_position_pre', 'ga4_sessions_pre', 'ga4_engaged_sessions_pre']


## 3. Verify it with exactly three warehouse queries

The first query checks the documented raw grain. The second checks the slice row count and date span. The third applies `IS TRUE` to the Search Console availability flag and shows how many rows survive. These are measured checks on the March 2026 partition, not assumptions from the documentation.

In [3]:
# Verification query 1 — raw grain: one row per report date × client × content.
verification_query_1 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS duplicate_rows
    FROM {REL}
    WHERE report_date = DATE '2026-03-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print('Duplicate raw-grain keys (expected 0 rows):', len(verification_query_1))
display(verification_query_1)

Duplicate raw-grain keys (expected 0 rows): 0


,report_date,client_hash_id,content_hash_id,duplicate_rows


In [4]:
# Verification query 2 — slice count and date span for March 2026.
verification_query_2 = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS content_items,
           MIN(report_date) AS min_report_date,
           MAX(report_date) AS max_report_date,
           COUNT(DISTINCT report_date) AS distinct_dates
    FROM {REL}
""").df()
display(verification_query_2)

,row_count,clients,content_items,min_report_date,max_report_date,distinct_dates
0,9841378,55,331437,2026-03-01,2026-03-31,31


In [5]:
# Verification query 3 — availability must be filtered with IS TRUE.
verification_query_3 = con.sql(f"""
    SELECT
        COUNT(*) AS all_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(DISTINCT client_hash_id) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_clients
    FROM {REL}
""").df()
display(verification_query_3)

,all_rows,gsc_available_rows,ga4_available_rows,gsc_available_clients
0,9841378,3611061,413966,47


### Five-feature frame and availability timing

The next query is a feature-building query rather than one of the three contract verification queries. It aggregates only the decision window and attaches a later-window proxy label. Each feature has an explicit availability moment: all five are computed from rows dated March 1–14, while the label uses March 15–31 and is not available at scoring time.

In [6]:
feature_sql = f"""
WITH per_pair AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' THEN gsc_impressions ELSE 0 END) AS gsc_impressions_pre,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' THEN gsc_clicks ELSE 0 END) AS gsc_clicks_pre,
        AVG(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS gsc_avg_position_pre,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' AND ga4_data_available IS TRUE THEN ga4_sessions ELSE NULL END) AS ga4_sessions_pre,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' AND ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE NULL END) AS ga4_engaged_sessions_pre,
        BOOL_OR(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' THEN gsc_data_available IS TRUE ELSE FALSE END) AS pre_gsc_available,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' THEN gsc_impressions ELSE 0 END) AS impressions_pre,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-15' AND DATE '2026-03-31' THEN gsc_impressions ELSE 0 END) AS impressions_eval
    FROM {REL}
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions_pre,
    gsc_clicks_pre,
    gsc_avg_position_pre,
    ga4_sessions_pre,
    ga4_engaged_sessions_pre,
    pre_gsc_available,
    impressions_pre,
    impressions_eval,
    (impressions_eval < impressions_pre) AS is_declining_proxy
FROM per_pair
WHERE pre_gsc_available IS TRUE
  AND impressions_pre > 0
"""
feature_frame = con.sql(feature_sql).df()
print('Feature frame shape:', feature_frame.shape)
print('Feature columns:', FEATURES)
display(feature_frame[['client_hash_id','content_hash_id'] + FEATURES + ['pre_gsc_available','is_declining_proxy']].head(10))
print('Proxy prevalence:', round(feature_frame['is_declining_proxy'].mean(), 4))

Feature frame shape: (150444, 11)
Feature columns: ['gsc_impressions_pre', 'gsc_clicks_pre', 'gsc_avg_position_pre', 'ga4_sessions_pre', 'ga4_engaged_sessions_pre']


,client_hash_id,content_hash_id,gsc_impressions_pre,gsc_clicks_pre,gsc_avg_position_pre,ga4_sessions_pre,ga4_engaged_sessions_pre,pre_gsc_available,is_declining_proxy
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4003.0,6.0,6.245228,NaN,NaN,True,True
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,231.0,0.0,4.185913,NaN,NaN,True,True
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3549.0,3.0,6.268105,NaN,NaN,True,True
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2395.0,8.0,7.202232,NaN,NaN,True,False
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,13.0,0.0,21.000000,NaN,NaN,True,False
5,client_73cda7b4e4f265ea,content_1855a661b4d36130,214.0,1.0,3.892111,NaN,NaN,True,False
6,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,122.0,0.0,9.527296,NaN,NaN,True,True
7,client_73cda7b4e4f265ea,content_1f380a642aed423b,43.0,1.0,11.008333,NaN,NaN,True,False
8,client_73cda7b4e4f265ea,content_22c063002b7c1caf,161.0,0.0,7.567988,NaN,NaN,True,True
9,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,2904.0,15.0,5.534299,NaN,NaN,True,False


Proxy prevalence: 0.3708


## 4. The deliberate leakage trap

I first add `is_declining_proxy` itself as a pretend feature and score by it. That score is intentionally invalid: it uses the answer to rank the answer, so it should look nearly perfect. I then delete the leaky column and keep the honest pre-decision feature list unchanged. The point is not to celebrate the leaked score; it is to show why a label-derived field must be removed before any model comparison.

In [7]:
# Deliberate leakage experiment: label itself as a feature.
def precision_at_k(frame, score_col, k=50):
    top = frame.sort_values(score_col, ascending=False).head(min(k, len(frame)))
    return float(top['is_declining_proxy'].mean()) if len(top) else float('nan')

leaky_frame = feature_frame.copy()
leaky_frame['LEAK_label_as_feature'] = leaky_frame['is_declining_proxy'].astype(int)
leaky_precision = precision_at_k(leaky_frame, 'LEAK_label_as_feature')

# An honest, label-free prioritization score: lower pre-window impressions means more review urgency.
honest_frame = feature_frame.copy()
honest_frame['honest_review_score'] = -honest_frame['gsc_impressions_pre'].astype(float)
honest_precision = precision_at_k(honest_frame, 'honest_review_score')

print('Leaky Precision@50 (invalid, expected to be near 1.0):', round(leaky_precision, 4))
print('Honest label-free Precision@50 (decision-time score):', round(honest_precision, 4))
print('The leaky column is now deleted from the modeling feature list.')
del leaky_frame['LEAK_label_as_feature']
FINAL_FEATURES = FEATURES
print('Final features retained:', FINAL_FEATURES)

Leaky Precision@50 (invalid, expected to be near 1.0): 1.0
Honest label-free Precision@50 (decision-time score): 0.42
The leaky column is now deleted from the modeling feature list.
Final features retained: ['gsc_impressions_pre', 'gsc_clicks_pre', 'gsc_avg_position_pre', 'ga4_sessions_pre', 'ga4_engaged_sessions_pre']


## Self-check

- [x] Five plain-words contract answers are written above and backed by real warehouse outputs.
- [x] Exactly three verification queries are shown: grain, counts/date span, and availability using `IS TRUE`.
- [x] The feature frame has five decision-time features and shows the client–content unit.
- [x] The label-derived leakage experiment is visible, scored, and removed from the final feature list.
- [x] The final-month `_sample` was not used; development uses the mid-panel March 2026 partition.
- [x] No client names, URLs, private queries, or access tokens are included in this notebook.

**Limitation.** This first slice is an unbalanced panel and the GA4 fields are unavailable for some client–content rows. The proxy is directional rather than causal: a lower later-window impression total does not prove that refreshing the page will restore performance. Any next model should use grouped client splits and a time-respecting evaluation window.